# Hints — Temporal Analytics

**Level:** Hard

> These hints are here when you're stuck. Try solving each problem on your own first.
> Hints are provided for selected problems - the trickier ones. If a problem is not listed here, it is meant to be solvable from the docs and earlier problems.
> Reveal one hint at a time.


## Problem 1


<details>
<summary>Hint 1 — using row_number to find the first transaction per customer</summary>

```python
from pyspark.sql import Window
w = Window.partitionBy('customerID').orderBy('dateTime')
labeled = transactions.withColumn('rn', F.row_number().over(w)) \
    .withColumn('purchase_type',
        F.when(F.col('rn') == 1, 'first_purchase').otherwise('repeat'))
```
`row_number()` restarts at 1 for each `customerID`. Row 1 (chronologically) = first purchase.

</details>


<details>
<summary>Hint 2 — aggregating the summary</summary>

```python
result_1 = labeled.groupBy('purchase_type').agg(
    F.count('*').alias('transaction_count'),
    F.round(F.sum('totalPrice'), 2).alias('total_revenue'),
    F.round(F.avg('totalPrice'), 2).alias('avg_order_value')
).orderBy(F.col('transaction_count').desc())
```

</details>


## Problem 2


<details>
<summary>Hint 1 — computing the gap with lag</summary>

```python
w = Window.partitionBy('customerID').orderBy('dateTime')
with_gap = transactions.withColumn(
    'prev_date', F.lag(F.to_date('dateTime')).over(w)
).withColumn(
    'gap_days', F.datediff(F.to_date('dateTime'), F.col('prev_date'))
)
```
`F.lag(..., 1)` returns the previous row's value — null on the very first row per customer.
That null naturally drops out of `avg` and `max` aggregations.

</details>


<details>
<summary>Hint 2 — aggregating and filtering</summary>

```python
result_2 = with_gap.groupBy('customerID').agg(
    F.count('*').alias('purchase_count'),
    F.round(F.avg('gap_days'), 1).alias('avg_gap_days'),
    F.max('gap_days').alias('max_gap_days')
).filter(F.col('purchase_count') >= 2) \
 .orderBy(F.col('max_gap_days').desc())
```

</details>


## Problem 3


<details>
<summary>Hint 1 — getting the reference date without current_date()</summary>

```python
# Compute once, collect as a Python value
ref_date = transactions.agg(
    F.to_date(F.max('dateTime')).alias('ref')
).collect()[0]['ref']

# Use F.lit() to broadcast it back into a column
from pyspark.sql.types import DateType
ref_col = F.lit(ref_date).cast(DateType())
```
Using the dataset's own max date makes the analysis reproducible regardless of when you run it.

</details>


<details>
<summary>Hint 2 — labelling and summarising</summary>

```python
per_customer = transactions.groupBy('customerID').agg(
    F.to_date(F.max('dateTime')).alias('last_purchase'),
    F.sum('totalPrice').alias('total_spend')
).withColumn('days_inactive', F.datediff(ref_col, F.col('last_purchase'))) \
 .withColumn('segment',
     F.when(F.col('days_inactive') <= 90, 'active').otherwise('lapsed'))

result_3 = per_customer.groupBy('segment').agg(
    F.count('*').alias('customer_count'),
    F.round(F.avg('total_spend'), 2).alias('avg_total_spend')
)
```

</details>


## Problem 4


<details>
<summary>Hint 1 — truncating to week and computing lag</summary>

```python
weekly = transactions.withColumn(
    'week_start', F.to_date(F.date_trunc('week', F.to_timestamp('dateTime')))
).groupBy('week_start').agg(
    F.round(F.sum('totalPrice'), 2).alias('weekly_revenue')
).orderBy('week_start')

w_lag = Window.orderBy('week_start')
weekly = weekly.withColumn(
    'prev_week_revenue', F.lag('weekly_revenue', 1).over(w_lag)
)
```
`F.date_trunc('week', ...)` returns the Monday of each week as a timestamp.

</details>


<details>
<summary>Hint 2 — computing WoW % change and trend label</summary>

```python
result_4 = weekly.withColumn(
    'wow_pct_change',
    F.round(
        (F.col('weekly_revenue') - F.col('prev_week_revenue'))
        / F.col('prev_week_revenue') * 100, 2
    )
).withColumn(
    'trend',
    F.when(F.col('wow_pct_change') > 10,  'growth')
     .when(F.col('wow_pct_change') < -10, 'decline')
     .when(F.col('wow_pct_change').isNotNull(), 'stable')
)
```
The final `.when(isNotNull, 'stable')` ensures the first week (null `wow_pct_change`) stays null
in `trend` rather than becoming `'stable'`.

</details>


## Problem 5


<details>
<summary>Hint 1 — assigning cohort months</summary>

```python
cohorts = transactions.groupBy('customerID').agg(
    F.date_trunc('month', F.min('dateTime')).alias('cohort_month')
)
# Join back so every transaction knows its customer's cohort
enriched = transactions.join(cohorts, on='customerID') \
    .withColumn('purchase_month', F.date_trunc('month', F.to_timestamp('dateTime')))
```

</details>


<details>
<summary>Hint 2 — computing months_since_cohort and retention_rate</summary>

```python
activity = enriched.groupBy('cohort_month', 'purchase_month').agg(
    F.countDistinct('customerID').alias('retained_customers')
).withColumn(
    'months_since_cohort',
    F.round(F.months_between('purchase_month', 'cohort_month')).cast('int')
)

# cohort_size = retained_customers at month 0
cohort_sizes = activity.filter(F.col('months_since_cohort') == 0) \
    .select('cohort_month', F.col('retained_customers').alias('cohort_size'))

result_5 = activity.join(cohort_sizes, on='cohort_month') \
    .withColumn('retention_rate',
        F.round(F.col('retained_customers') / F.col('cohort_size') * 100, 1)) \
    .orderBy('cohort_month', 'months_since_cohort')
```

</details>


## Problem 6


<details>
<summary>Hint 1 — why the date − row_number trick works</summary>

If a customer bought on days **3, 4, 5, 8**:

| rn | purchase_date | date − rn |
|----|--------------|----------|
| 1  | day 3        | day 2    |
| 2  | day 4        | day 2    |
| 3  | day 5        | day 2    |
| 4  | day 8        | day 4    |

Days 3, 4, 5 all produce `day 2` → same island (streak of 3). Day 8 produces `day 4` → different island.

</details>


<details>
<summary>Hint 2 — full implementation</summary>

```python
# Step 1: distinct purchase dates per customer
daily = transactions.select(
    'customerID',
    F.to_date('dateTime').alias('purchase_date')
).distinct()

# Step 2: row_number per customer ordered by date
w = Window.partitionBy('customerID').orderBy('purchase_date')
daily = daily.withColumn('rn', F.row_number().over(w))

# Step 3: island key
daily = daily.withColumn(
    'island_key', F.date_sub('purchase_date', F.col('rn').cast('int'))
)

# Step 4: streak length per island
streaks = daily.groupBy('customerID', 'island_key').agg(
    F.count('*').alias('streak_length')
)

# Step 5: longest streak per customer
result_6 = streaks.groupBy('customerID').agg(
    F.max('streak_length').alias('longest_streak_days')
).orderBy(F.col('longest_streak_days').desc())
```

</details>
